# El Detective de Señales
## Clasificación automática de modulaciones con Deep Learning

**Alumno:** Diego Sanchez Gutierrez  
**Institución:** Tecnológico de Estudios Superiores de Ecatepec (TESE)  
**Asignatura:** Telecomunicaciones  
**Proyecto:** Generación, entrenamiento y evaluación de señales moduladas (Fases 1, 2 y 3)

### Descripción general
El proyecto consiste en desarrollar un clasificador automático de señales de telecomunicaciones mediante una red neuronal convolucional unidimensional (CNN 1D). Se consideran cuatro esquemas de modulación: ASK, FSK, PSK y 16-QAM. Las señales se simulan con distintos niveles de ruido gaussiano blanco aditivo (AWGN) para analizar el comportamiento del clasificador en condiciones de transmisión variadas.

El desarrollo se organiza en tres etapas: generación y preparación de señales, entrenamiento de la red neuronal y evaluación de su precisión y robustez frente al ruido. Los resultados se obtienen a partir de la ejecución del código y se presentan mediante tablas y gráficas.

## Fase 1. Generación y preparación de datos

Se generan **4,000 señales**, distribuidas en 1,000 ejemplos de cada modulación. Cada señal contiene 1,024 muestras y se combina con uno de seis niveles de ruido AWGN. El conjunto se divide en 3,200 señales para entrenamiento y validación y 800 señales destinadas exclusivamente a la evaluación final.

### 1.1 Configuración del entorno

El entorno de trabajo está desarrollado en Python. Se utilizan **NumPy** para los cálculos numéricos, **Pandas** para organizar los registros, **Matplotlib** para representar las señales, **SciPy** para el análisis espectral y **Scikit-learn** para separar los conjuntos de datos. Se fija una semilla aleatoria para facilitar la reproducción de los experimentos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import welch
from sklearn.model_selection import train_test_split

SEMILLA = 2026
rng = np.random.default_rng(SEMILLA)
print("Entorno listo. NumPy:", np.__version__)

### 1.2 Parámetros de las señales

| Parámetro | Valor | Significado |
|---|---:|---|
| $f_s$ | 8,000 Hz | Frecuencia de muestreo |
| $f_c$ | 1,600 Hz | Frecuencia portadora |
| $R_s$ | 250 símbolos/s | Tasa de símbolos |
| $N_s$ | 32 | Símbolos por señal |
| $f_s / R_s$ | 32 | Muestras por símbolo |
| $N$ | 1,024 | Longitud de cada señal (32 × 32) |
| $\Delta f$ | 250 Hz | Desviación de frecuencia en FSK |

La duración de cada señal es de $N/f_s=0.128$ segundos. La frecuencia de muestreo satisface el criterio de Nyquist para las frecuencias consideradas. Se implementan **2-ASK (OOK), 2-FSK, BPSK y 16-QAM**.

In [ ]:
FS = 8000                   # Hz
FC = 1600                   # Hz
RS = 250                    # simbolos por segundo
MUESTRAS_POR_SIMBOLO = FS // RS
NUM_SIMBOLOS = 32
N = NUM_SIMBOLOS * MUESTRAS_POR_SIMBOLO
DELTA_F = 250              # Hz para FSK
MODULACIONES = ["ASK", "FSK", "PSK", "QAM"]
N_POR_CLASE = 1000
NIVELES_SNR = np.array([-5, 0, 5, 10, 15, 20], dtype=np.int16)

assert FS % RS == 0
assert FC + DELTA_F < FS / 2
print(f"{MUESTRAS_POR_SIMBOLO} muestras por simbolo | {N} puntos por señal | {N/FS:.3f} s")

### 1.3 Generación de señales moduladas

La modulación cambia una característica de la señal portadora para representar los símbolos digitales:

- **ASK (On-Off Keying):** modifica la amplitud mediante valores de 0 y 1.
- **FSK:** utiliza dos frecuencias, 1,350 Hz y 1,850 Hz, según el bit transmitido.
- **PSK (BPSK):** representa los bits mediante dos fases separadas 180°.
- **16-QAM:** combina cuatro niveles de las componentes en fase ($I$) y cuadratura ($Q$), obteniendo 16 combinaciones.

Las señales pueden describirse mediante las siguientes expresiones:

$$s_{ASK}(t)=A_k\cos(2\pi f_ct+\phi)$$
$$s_{FSK}(t)=\cos\left(2\pi\int_0^t f(\tau)\,d\tau+\phi\right)$$
$$s_{PSK}(t)=b_k\cos(2\pi f_ct+\phi),\quad b_k\in\{-1,+1\}$$
$$s_{QAM}(t)=I_k\cos(2\pi f_ct+\phi)-Q_k\sin(2\pi f_ct+\phi)$$

Cada ejemplo se construye con símbolos aleatorios y una fase inicial variable. Después se normaliza para obtener una potencia media cercana a la unidad, de modo que el clasificador no dependa únicamente de diferencias globales de energía.

In [ ]:
def generar_senal_limpia(tipo, generador):
    """Genera una señal real de N muestras, modulada y sin ruido."""
    bits = generador.integers(0, 2, size=NUM_SIMBOLOS)
    fase_inicial = generador.uniform(0, 2 * np.pi)
    t = np.arange(N, dtype=np.float64) / FS
    portadora = 2 * np.pi * FC * t + fase_inicial

    if tipo == "ASK":
        amplitudes = np.repeat(bits, MUESTRAS_POR_SIMBOLO)
        s = amplitudes * np.cos(portadora)

    elif tipo == "FSK":
        frecuencias_por_simbolo = FC + DELTA_F * (2 * bits - 1)
        frecuencia_instantanea = np.repeat(frecuencias_por_simbolo,
                                         MUESTRAS_POR_SIMBOLO)
        # Integracion discreta de frecuencia: fase continua al cambiar de bit.
        fase = fase_inicial + np.concatenate((
            [0.0], 2 * np.pi * np.cumsum(frecuencia_instantanea[:-1]) / FS
        ))
        s = np.cos(fase)

    elif tipo == "PSK":
        simbolos = 2 * bits - 1  # BPSK: valores -1 o +1
        s = np.repeat(simbolos, MUESTRAS_POR_SIMBOLO) * np.cos(portadora)

    elif tipo == "QAM":
        niveles = np.array([-3, -1, 1, 3])
        i_simbolos = generador.choice(niveles, size=NUM_SIMBOLOS)
        q_simbolos = generador.choice(niveles, size=NUM_SIMBOLOS)
        i = np.repeat(i_simbolos, MUESTRAS_POR_SIMBOLO) / np.sqrt(10)
        q = np.repeat(q_simbolos, MUESTRAS_POR_SIMBOLO) / np.sqrt(10)
        s = i * np.cos(portadora) - q * np.sin(portadora)

    else:
        raise ValueError(f"Modulación desconocida: {tipo}")

    potencia = np.mean(s ** 2)
    return (s / np.sqrt(potencia + 1e-12)).astype(np.float32)

# Verificar que las cuatro señales tengan la longitud correcta.
for tipo in MODULACIONES:
    muestra = generar_senal_limpia(tipo, rng)
    print(f"{tipo}: forma={muestra.shape}, potencia={np.mean(muestra**2):.3f}")

#### Comparación de las formas de onda

Las gráficas presentan los primeros cuatro símbolos de cada tipo de modulación. En ASK destacan los cambios de amplitud; en FSK, la variación de frecuencia; en PSK, las inversiones de fase; y en QAM, la combinación de cambios de amplitud y fase.

In [ ]:
fig, axs = plt.subplots(4, 1, figsize=(12, 9), sharex=True)
M = 4 * MUESTRAS_POR_SIMBOLO
tiempo_ms = np.arange(M) / FS * 1000

for ax, tipo in zip(axs, MODULACIONES):
    s = generar_senal_limpia(tipo, rng)
    ax.plot(tiempo_ms, s[:M], linewidth=1.3)
    ax.set_ylabel(f"{tipo}\nAmplitud")
    ax.grid(alpha=0.3)
    for x in np.arange(1, 4) * 1000 / RS:
        ax.axvline(x, linestyle="--", alpha=0.35, color="gray")

axs[-1].set_xlabel("Tiempo (ms)")
fig.suptitle("Señales limpias: ASK, FSK, PSK y 16-QAM", fontsize=14)
fig.tight_layout()
fig.savefig("vista_previa_senales.png", dpi=160, bbox_inches="tight")
plt.show()

### 1.5 Incorporación de ruido AWGN

El ruido gaussiano blanco aditivo se suma a las señales para simular alteraciones del canal. La **SNR** (relación señal-ruido) expresa la proporción entre la potencia de la señal y la del ruido:

$$SNR_{dB}=10\log_{10}\left(\frac{P_{señal}}{P_{ruido}}\right)$$

Una SNR baja, como −5 dB, representa condiciones con mayor presencia de ruido; una SNR alta, como 20 dB, permite conservar mejor la forma de la señal. Las gráficas permiten comparar ambos casos con la señal original.

In [ ]:
def agregar_ruido_awgn(senal, snr_db, generador):
    """Suma ruido normal(0,sigma) con la SNR objetivo en dB."""
    potencia_senal = float(np.mean(senal.astype(np.float64) ** 2))
    potencia_ruido = potencia_senal / (10 ** (float(snr_db) / 10))
    desviacion = np.sqrt(potencia_ruido)
    ruido = generador.normal(0, desviacion, size=senal.shape)
    return (senal + ruido).astype(np.float32)

senal_referencia = generar_senal_limpia("FSK", rng)
fig, axs = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
for ax, nombre, snr in zip(
    axs,
    ["FSK sin ruido", "FSK con SNR = 20 dB", "FSK con SNR = -5 dB"],
    [None, 20, -5]
):
    s = senal_referencia if snr is None else agregar_ruido_awgn(senal_referencia, snr, rng)
    ax.plot(tiempo_ms, s[:M], linewidth=1.0)
    ax.set_title(nombre)
    ax.set_ylabel("Amplitud")
    ax.grid(alpha=0.25)
axs[-1].set_xlabel("Tiempo (ms)")
fig.tight_layout()
fig.savefig("efecto_ruido_awgn.png", dpi=160, bbox_inches="tight")
plt.show()

### 1.4 Construcción del conjunto de datos

Se generan **1,000 señales por modulación**, para un total de **4,000 ejemplos independientes**. Las SNR de −5, 0, 5, 10, 15 y 20 dB se distribuyen de manera aproximadamente uniforme entre las cuatro clases.

- `X`: matriz de amplitudes, de tamaño **(4000, 1024)**.
- `y`: etiquetas de clase (**0 = ASK, 1 = FSK, 2 = PSK, 3 = QAM**).
- `snr_db`: nivel de ruido correspondiente a cada ejemplo.

La generación independiente de señales reduce el riesgo de que formas de onda casi idénticas aparezcan en los conjuntos de entrenamiento y prueba.

In [ ]:
TOTAL = N_POR_CLASE * len(MODULACIONES)
X = np.empty((TOTAL, N), dtype=np.float32)
y = np.empty(TOTAL, dtype=np.int64)
snr_db = np.empty(TOTAL, dtype=np.int16)

posicion = 0
for clase_id, tipo in enumerate(MODULACIONES):
    # Se reparten los seis niveles y se mezclan en cada clase.
    snr_de_clase = np.resize(NIVELES_SNR, N_POR_CLASE)
    rng.shuffle(snr_de_clase)
    for nivel in snr_de_clase:
        limpia = generar_senal_limpia(tipo, rng)
        X[posicion] = agregar_ruido_awgn(limpia, int(nivel), rng)
        y[posicion] = clase_id
        snr_db[posicion] = nivel
        posicion += 1

print("X (ondas):", X.shape)
print("y (clases):", y.shape)
print("snr_db:", snr_db.shape)
print("Tipo de datos:", X.dtype)
print("Muestras por clase:")
for idx, tipo in enumerate(MODULACIONES):
    print(f"  {tipo}: {np.sum(y == idx)}")
assert posicion == 4000
assert X.shape == (4000, 1024)
assert np.isfinite(X).all()

### 1.6 División de los datos

La separación se establece en **80 % para entrenamiento** y **20 % para prueba**. Por cada modulación se asignan 800 ejemplos al primer conjunto y 200 al segundo. Se aplica una división estratificada por SNR para mantener una distribución similar de los niveles de ruido. Las muestras de prueba permanecen fuera del ajuste de la red neuronal.

In [ ]:
indices_entrenamiento = []
indices_prueba = []

for clase_id in range(len(MODULACIONES)):
    indices_clase = np.flatnonzero(y == clase_id)
    ent, pru = train_test_split(
        indices_clase,
        test_size=0.20,
        random_state=SEMILLA + clase_id,
        stratify=snr_db[indices_clase]
    )
    indices_entrenamiento.extend(ent)
    indices_prueba.extend(pru)

# Mezclar el orden de las clases después de separar.
mezclador = np.random.default_rng(SEMILLA + 100)
indices_entrenamiento = mezclador.permutation(indices_entrenamiento)
indices_prueba = mezclador.permutation(indices_prueba)

X_train = X[indices_entrenamiento]
y_train = y[indices_entrenamiento]
snr_train = snr_db[indices_entrenamiento]
X_test = X[indices_prueba]
y_test = y[indices_prueba]
snr_test = snr_db[indices_prueba]

assert len(np.intersect1d(indices_entrenamiento, indices_prueba)) == 0
assert len(X_train) == 3200 and len(X_test) == 800

print("ENTRENAMIENTO:", X_train.shape, "etiquetas", y_train.shape)
print("PRUEBA:", X_test.shape, "etiquetas", y_test.shape)
print("\nCantidad por modulación:")
for clase_id, tipo in enumerate(MODULACIONES):
    print(f"{tipo:3s}: entrenamiento={np.sum(y_train == clase_id)}, "
          f"prueba={np.sum(y_test == clase_id)}")

### Distribución del conjunto generado

La tabla cruza las clases de modulación con los niveles de SNR. Permite comprobar que las cuatro clases están equilibradas, con **1,000 señales por modulación**.

In [ ]:
resumen = pd.crosstab(
    index=pd.Series(np.array(MODULACIONES)[y], name="Modulación"),
    columns=pd.Series(snr_db, name="SNR (dB)")
)
resumen["TOTAL"] = resumen.sum(axis=1)
print(resumen.to_string())

### Análisis espectral

Los espectros de potencia muestran cómo se distribuye la energía de las señales en función de la frecuencia. La comparación de ASK, FSK, PSK y 16-QAM complementa el análisis de sus formas de onda en el dominio temporal.

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(12, 7), sharex=True)
for ax, tipo in zip(axs.flat, MODULACIONES):
    s = generar_senal_limpia(tipo, rng)
    freq, potencia = welch(s, fs=FS, nperseg=512)
    ax.plot(freq, 10 * np.log10(potencia + 1e-15))
    ax.set_title(tipo if tipo != "QAM" else "16-QAM")
    ax.set_xlim(500, 2700)
    ax.set_xlabel("Frecuencia (Hz)")
    ax.set_ylabel("Potencia espectral (dB/Hz)")
    ax.grid(alpha=0.25)
fig.suptitle("Espectros de potencia (señales limpias)", fontsize=14)
fig.tight_layout()
fig.savefig("espectros_modulaciones.png", dpi=160, bbox_inches="tight")
plt.show()

### Almacenamiento del conjunto de datos

Las señales y etiquetas de entrenamiento y prueba se almacenan en `datos_senales.npz`; sus identificadores, clases y niveles de SNR se registran en `metadatos_senales.csv`. También se generan imágenes PNG de las formas de onda, la comparación con AWGN y los espectros de potencia. Estos archivos constituyen los resultados de la primera fase.

In [ ]:
np.savez_compressed(
    "datos_senales.npz",
    X_train=X_train, y_train=y_train, snr_train=snr_train,
    X_test=X_test, y_test=y_test, snr_test=snr_test,
    modulaciones=np.array(MODULACIONES),
    clases=np.array(MODULACIONES),
    fs=FS, fc=FC, rs=RS,
    num_simbolos=NUM_SIMBOLOS, muestras_por_simbolo=MUESTRAS_POR_SIMBOLO,
    niveles_snr=NIVELES_SNR, semilla=SEMILLA
)

metadata_train = pd.DataFrame({
    "id": [f"train_{i:04d}" for i in range(len(X_train))],
    "conjunto": "entrenamiento",
    "etiqueta_numerica": y_train,
    "modulacion": np.array(MODULACIONES)[y_train],
    "snr_db": snr_train
})
metadata_test = pd.DataFrame({
    "id": [f"test_{i:04d}" for i in range(len(X_test))],
    "conjunto": "prueba",
    "etiqueta_numerica": y_test,
    "modulacion": np.array(MODULACIONES)[y_test],
    "snr_db": snr_test
})
pd.concat([metadata_train, metadata_test], ignore_index=True).to_csv(
    "metadatos_senales.csv", index=False, encoding="utf-8-sig"
)

with np.load("datos_senales.npz") as archivo:
    print("Archivo guardado y verificado:")
    print("Entrenamiento:", archivo["X_train"].shape)
    print("Prueba:", archivo["X_test"].shape)
    print("Clases:", archivo["clases"].tolist())
print("Metadatos guardados: metadatos_senales.csv")


---
## Fase 2. Diseño y entrenamiento del modelo

Se desarrolla una **CNN 1D** para clasificar las cuatro modulaciones. De las 3,200 señales disponibles para entrenamiento se destinan **2,560 al ajuste de los parámetros** y **640 a validación**. Las 800 muestras de prueba se mantienen separadas para la evaluación de la Fase 3. Los datos utilizados corresponden al archivo generado durante la primera fase.

### 2.1 Fundamento de la arquitectura CNN

Las redes neuronales convolucionales utilizan filtros para identificar patrones locales. En señales temporales, estos patrones pueden corresponder a cambios de amplitud, diferencias de frecuencia o transiciones de fase.

Se analizaron dos enfoques: **CNN 1D**, que trabaja directamente con las muestras temporales, y **CNN 2D**, que procesa representaciones como espectrogramas. Se eligió **CNN 1D** por la estructura original de los datos: cada señal está representada por un vector de **1,024 muestras**. Esta elección conserva el orden temporal y evita una conversión previa a imágenes.

Referencias: [Conv1D de Keras](https://keras.io/api/layers/convolution_layers/convolution1d/) y [entrenamiento con Keras](https://keras.io/guides/training_with_built_in_methods/).

### Preparación de bibliotecas y datos

Para el entrenamiento se emplean **TensorFlow/Keras**, **NumPy**, **Pandas**, **Matplotlib** y **Scikit-learn**. El conjunto de señales se recupera del archivo `datos_senales.npz`, conservando las etiquetas y la división realizada en la Fase 1.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

np.random.seed(15)
print("Bibliotecas para preparar los datos: listas")

In [ ]:
import os

if not os.path.isfile("datos_senales.npz"):
    raise FileNotFoundError(
        "No se encontró datos_senales.npz, requerido para el entrenamiento."
    )
print("Archivo de la Fase 1 encontrado: datos_senales.npz")
print("Datos de la Fase 1 recuperados para entrenamiento y evaluación.")

### Preparación de entradas y validación

Las **3,200 señales** iniciales se dividen en **2,560 ejemplos de entrenamiento efectivo** y **640 de validación**. El conjunto de prueba conserva **800 señales**, utilizadas únicamente en la Fase 3.

| Conjunto | Número de señales |
|---|---:|
| Entrenamiento | 2,560 |
| Validación | 640 |
| Prueba | 800 |

Las señales se reorganizan con dimensiones **(1024, 1)** para su procesamiento en `Conv1D`. Las etiquetas se codifican en cuatro componentes mediante representación *one-hot*; por ejemplo, ASK corresponde a `[1, 0, 0, 0]`.

In [ ]:
with np.load("datos_senales.npz", allow_pickle=False) as datos:
    X_train = datos["X_train"].astype(np.float32)
    y_train = datos["y_train"].astype(int)
    snr_train = datos["snr_train"]
    X_test = datos["X_test"].astype(np.float32)
    y_test = datos["y_test"].astype(int)
    snr_test = datos["snr_test"]
    modulaciones = datos["modulaciones"].tolist()

assert modulaciones == ["ASK", "FSK", "PSK", "QAM"]
assert X_train.shape == (3200, 1024) and X_test.shape == (800, 1024)
assert len(y_train) == 3200 and len(y_test) == 800

# Mantengo la proporción de cada modulación y nivel de ruido
grupos = [f"{clase}_{snr}" for clase, snr in zip(y_train, snr_train)]
ind_ent, ind_val = train_test_split(
    np.arange(len(y_train)), test_size=0.20,
    random_state=15, stratify=grupos
)

X_ent = X_train[ind_ent, :, np.newaxis]
X_val = X_train[ind_val, :, np.newaxis]
y_ent = np.eye(4, dtype=np.float32)[y_train[ind_ent]]
y_val = np.eye(4, dtype=np.float32)[y_train[ind_val]]

print("Entrenamiento:", X_ent.shape)
print("Validación:", X_val.shape)
print("Prueba reservada:", X_test.shape)

### 2.2 Arquitectura de la red neuronal

Se utiliza una entrada de **1,024 muestras y un canal**. Las capas convolucionales identifican características de las señales; **ReLU** introduce no linealidad y **MaxPooling1D** reduce progresivamente las dimensiones. La capa **Flatten** transforma los mapas de características en un vector. Una capa densa integra la información y **Softmax** calcula la probabilidad de pertenecer a cada una de las cuatro modulaciones.

| Capa | Función |
|---|---|
| Entrada | 1,024 muestras × 1 canal |
| Conv1D (32 filtros) + ReLU | Detecta patrones locales |
| MaxPooling1D | Reduce la longitud a 256 |
| Conv1D (64 filtros) + ReLU | Extrae características más complejas |
| MaxPooling1D | Reduce la longitud a 32 |
| Flatten | Construye un vector de características |
| Dense (64) + ReLU | Combina la información extraída |
| Dropout | Reduce el riesgo de sobreajuste |
| Dense (4) + Softmax | Produce las probabilidades de ASK, FSK, PSK y QAM |

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

keras.utils.set_random_seed(15)
print("TensorFlow:", tf.__version__)
print("GPU disponible:", bool(tf.config.list_physical_devices("GPU")))

modelo = keras.Sequential([
    keras.Input(shape=(1024, 1)),
    layers.Conv1D(32, kernel_size=17, activation="relu", padding="same"),
    layers.MaxPooling1D(pool_size=4),
    layers.Conv1D(64, kernel_size=9, activation="relu", padding="same"),
    layers.MaxPooling1D(pool_size=8),
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.25),
    layers.Dense(4, activation="softmax")
])

modelo.summary()

### 2.3 Compilación del modelo

La red se compila con el optimizador **Adam**, que actualiza los pesos durante el aprendizaje. Se utiliza **categorical crossentropy** para medir la diferencia entre las probabilidades predichas y las etiquetas reales. La métrica **accuracy** representa la proporción de clasificaciones correctas.

In [ ]:
modelo.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
print("Modelo compilado correctamente")

### 2.4 Entrenamiento

Se configura un máximo de **25 épocas** con lotes de **32 señales**. Durante cada época se registran la precisión (`accuracy`), la pérdida (`loss`) y sus valores de validación (`val_accuracy` y `val_loss`).

Se utiliza `ModelCheckpoint` para conservar automáticamente el modelo con menor pérdida de validación y `EarlyStopping` para detener el entrenamiento cuando deja de observarse una mejora sostenida. Esta estrategia permite reducir el sobreajuste.

In [ ]:
epocas = 25
lote = 32

guardar_mejor = keras.callbacks.ModelCheckpoint(
    "modelo_senales.keras", monitor="val_loss",
    save_best_only=True, verbose=0
)
detener = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=6, restore_best_weights=True
)

historial = modelo.fit(
    X_ent, y_ent,
    validation_data=(X_val, y_val),
    epochs=epocas,
    batch_size=lote,
    callbacks=[guardar_mejor, detener],
    verbose=1
)

### Evolución de la precisión y de la pérdida

Las gráficas representan la precisión y la pérdida de entrenamiento y validación a lo largo de las épocas. La comparación permite identificar el avance del aprendizaje y posibles diferencias entre el desempeño con datos utilizados para ajustar la red y datos de validación.

In [ ]:
registro = pd.DataFrame(historial.history)
registro.index = np.arange(1, len(registro) + 1)
registro.index.name = "epoca"

fig, axs = plt.subplots(1, 2, figsize=(12, 4))

axs[0].plot(registro.index, registro["accuracy"], label="Entrenamiento")
axs[0].plot(registro.index, registro["val_accuracy"], label="Validación")
axs[0].set_title("Precisión por época")
axs[0].set_xlabel("Época")
axs[0].set_ylabel("Accuracy")
axs[0].set_ylim(0, 1.05)
axs[0].legend()
axs[0].grid(alpha=0.3)

axs[1].plot(registro.index, registro["loss"], label="Entrenamiento")
axs[1].plot(registro.index, registro["val_loss"], label="Validación")
axs[1].set_title("Pérdida por época")
axs[1].set_xlabel("Época")
axs[1].set_ylabel("Loss")
axs[1].legend()
axs[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("graficas_entrenamiento.png", dpi=150)
plt.show()

mejor_epoca = registro["val_loss"].idxmin()
print("Mejor época según la pérdida de validación:", mejor_epoca)
print(f"Precisión de validación en esa época: {registro.loc[mejor_epoca, 'val_accuracy']:.2%}")

### 2.5 Conservación del modelo entrenado

El archivo `modelo_senales.keras` almacena la arquitectura y los pesos de la red neuronal. También se conserva el historial de entrenamiento en `historial_entrenamiento.csv` y la gráfica correspondiente en `graficas_entrenamiento.png`. El modelo se vuelve a cargar para comprobar que su estructura se mantuvo correctamente.

In [ ]:
modelo = keras.models.load_model("modelo_senales.keras")
# Conservar el mejor modelo según la pérdida de validación
modelo.save("modelo_senales.keras")
registro.to_csv("historial_entrenamiento.csv")

modelo_guardado = keras.models.load_model("modelo_senales.keras")
assert modelo_guardado.input_shape == (None, 1024, 1)
assert modelo_guardado.output_shape == (None, 4)

print("Modelo guardado: modelo_senales.keras")
print("Historial: historial_entrenamiento.csv")
print("Gráficas: graficas_entrenamiento.png")

### Ejemplo de clasificación

Se selecciona una señal del conjunto de validación y se obtiene la probabilidad asignada por el modelo a cada modulación. El resultado permite comparar la clase real con la clase predicha y visualizar la distribución de probabilidades de salida.

In [ ]:
ejemplo = 0
probabilidades = modelo_guardado.predict(X_val[ejemplo:ejemplo+1], verbose=0)[0]
real = modulaciones[np.argmax(y_val[ejemplo])]
prediccion = modulaciones[np.argmax(probabilidades)]

print("Modulación real:", real)
print("Predicción:", prediccion)

fig, axs = plt.subplots(1, 2, figsize=(11, 4))
axs[0].plot(np.arange(128) / 8000 * 1000, X_val[ejemplo, :128, 0])
axs[0].set_title("Señal de validación")
axs[0].set_xlabel("Tiempo (ms)")
axs[0].set_ylabel("Amplitud")

axs[1].bar(modulaciones, probabilidades * 100, color="steelblue")
axs[1].set_title("Probabilidades del modelo")
axs[1].set_ylabel("Probabilidad (%)")
axs[1].set_ylim(0, 100)
plt.tight_layout()
plt.show()

### Resultados de la Fase 2

El proceso produce un modelo entrenado (`modelo_senales.keras`), un historial de métricas (`historial_entrenamiento.csv`) y una gráfica de precisión y pérdida (`graficas_entrenamiento.png`). El mejor estado del modelo se selecciona en función de la pérdida de validación.

La evaluación definitiva de clasificación y robustez se realiza por separado sobre las 800 señales de prueba. Este procedimiento distingue el ajuste del modelo de la medición de su desempeño con datos reservados.

Referencia: [Keras, guardado y carga de modelos](https://keras.io/api/models/model_saving_apis/model_saving_and_loading/).

---
## Fase 3. Evaluación y robustez frente al ruido

Se evalúa el modelo entrenado mediante las **800 señales del conjunto de prueba**, sin realizar ajustes adicionales a los pesos. Esta fase incluye una matriz de confusión, el cálculo de precisión para distintos niveles de SNR, gráficas de rendimiento y un análisis de los principales errores de clasificación.

In [ ]:
import os
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (accuracy_score, confusion_matrix,
                             ConfusionMatrixDisplay, classification_report)
from tensorflow import keras

print("Bibliotecas cargadas correctamente")

### Carga del modelo y de las señales de prueba

In [ ]:
archivos_necesarios = ["modelo_senales.keras", "datos_senales.npz"]
faltan = [nombre for nombre in archivos_necesarios if not os.path.isfile(nombre)]
if faltan:
    raise FileNotFoundError(
        "No se encontraron " + ", ".join(faltan) +
        ". El modelo y las señales de prueba son necesarios para esta evaluación."
    )
print("Modelo entrenado y señales de prueba disponibles.")

In [ ]:
with np.load("datos_senales.npz", allow_pickle=False) as datos:
    campos = {"X_test", "y_test", "snr_test", "modulaciones"}
    if not campos.issubset(datos.files):
        raise ValueError(
            "El conjunto de datos no tiene el formato esperado. "
            "Faltan los campos utilizados durante el entrenamiento."
        )
    X_test = datos["X_test"].astype(np.float32)
    y_test = datos["y_test"].astype(int)
    snr_test = datos["snr_test"].astype(int)
    modulaciones = [str(x) for x in datos["modulaciones"].tolist()]

if modulaciones != ["ASK", "FSK", "PSK", "QAM"]:
    raise ValueError("Las clases no coinciden con las de la Fase 2")
if X_test.shape != (800, 1024) or y_test.shape != (800,) or snr_test.shape != (800,):
    raise ValueError("Las dimensiones de prueba no son las esperadas (800 señales de 1024 muestras)")
if not np.isfinite(X_test).all() or not np.isin(y_test, range(4)).all():
    raise ValueError("El conjunto de prueba tiene valores incorrectos")

modelo = keras.models.load_model("modelo_senales.keras", compile=False)
if tuple(modelo.input_shape[1:]) != (1024, 1) or tuple(modelo.output_shape[1:]) != (4,):
    raise ValueError("El modelo no corresponde a la arquitectura de la Fase 2")

X_entrada = X_test[:, :, np.newaxis]
print("Modelo cargado:", modelo.name)
print("Señales de prueba:", X_entrada.shape)
print("Distribución por clase:", {m: int(np.sum(y_test == i)) for i, m in enumerate(modulaciones)})
print("SNR disponibles (dB):", sorted(np.unique(snr_test).tolist()))
print("No se ha entrenado el modelo en esta fase.")

### 3.1 Matriz de confusión

Se comparan las etiquetas reales y las predicciones de la CNN. En la matriz de confusión, la diagonal principal corresponde a los aciertos y los valores fuera de ella a las clasificaciones incorrectas. Se calcula también una versión normalizada en porcentajes para facilitar la comparación entre clases.

Además, se registran específicamente las confusiones entre **ASK y PSK**, debido a que las alteraciones introducidas por el ruido pueden afectar la identificación de sus características.

In [ ]:
probabilidades = modelo.predict(X_entrada, batch_size=64, verbose=0)
if probabilidades.shape != (len(y_test), 4):
    raise ValueError("La red debe dar cuatro probabilidades por señal")
y_pred = np.argmax(probabilidades, axis=1)
exactitud = accuracy_score(y_test, y_pred)

matriz = confusion_matrix(y_test, y_pred, labels=np.arange(4))
matriz_porcentaje = matriz / matriz.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.9))
ConfusionMatrixDisplay(matriz, display_labels=modulaciones).plot(
    ax=axes[0], cmap="Blues", colorbar=False, values_format="d"
)
axes[0].set_title("Matriz de confusión (cantidad)")
ConfusionMatrixDisplay(matriz_porcentaje, display_labels=modulaciones).plot(
    ax=axes[1], cmap="Blues", colorbar=False, values_format=".1%"
)
axes[1].set_title("Matriz de confusión (porcentaje)")
fig.tight_layout()
fig.savefig("matriz_confusion_fase3.png", dpi=160, bbox_inches="tight")
plt.show()

pd.DataFrame(matriz, index=modulaciones, columns=modulaciones).to_csv(
    "matriz_confusion_fase3.csv", index_label="Clase real"
)
print(f"Precisión total en prueba: {exactitud * 100:.2f}% ({int(np.trace(matriz))}/800)")
print("Filas = clase real; columnas = clase predicha")

In [ ]:
ask_como_psk = int(matriz[0, 2])
psk_como_ask = int(matriz[2, 0])
print(f"ASK confundidas con PSK: {ask_como_psk}")
print(f"PSK confundidas con ASK: {psk_como_ask}")

if ask_como_psk + psk_como_ask > 0:
    print("Sí hubo confusiones ASK/PSK en este conjunto de prueba.")
    print("Hipótesis: el ruido dificulta distinguir cambios de amplitud y de fase en algunas formas de onda.")
else:
    print("No hubo confusiones ASK/PSK en estas 800 señales de prueba.")
    print("Esto no garantiza que nunca ocurran con otras señales o niveles de ruido.")

### 3.2 Evaluación con diferentes niveles de ruido

Las señales de prueba incluyen SNR de **−5, 0, 5, 10, 15 y 20 dB**. Los resultados se agrupan en tres categorías: **ruido alto** (−5 y 0 dB), **intermedio** (5 y 10 dB) y **ruido bajo** (15 y 20 dB). Se calcula la cantidad de aciertos y la precisión para cada nivel y grupo, sin modificar el modelo.

In [ ]:
filas = []
for snr in sorted(np.unique(snr_test)):
    indices = snr_test == snr
    n = int(indices.sum())
    aciertos = int(np.sum(y_pred[indices] == y_test[indices]))
    filas.append({"SNR (dB)": int(snr), "Señales": n,
                  "Aciertos": aciertos, "Precisión (%)": round(100 * aciertos / n, 2)})

resultados_snr = pd.DataFrame(filas)
print("RESULTADOS POR NIVEL DE RUIDO")
print(resultados_snr.to_string(index=False))
resultados_snr.to_csv("precision_por_snr.csv", index=False)

print("\nRESULTADOS POR GRUPO")
grupos_ruido = {
    "Bajo (−5 y 0 dB)": [-5, 0],
    "Medio (5 y 10 dB)": [5, 10],
    "Alto (15 y 20 dB)": [15, 20],
}
for grupo, niveles in grupos_ruido.items():
    indices = np.isin(snr_test, niveles)
    cantidad = int(indices.sum())
    if cantidad:
        acc = 100 * np.mean(y_pred[indices] == y_test[indices])
        print(f"{grupo}: {acc:.2f}% de aciertos ({cantidad} señales)")

### 3.3 Rendimiento en función de la SNR

La gráfica relaciona el nivel de SNR en el eje horizontal con la precisión de clasificación en el eje vertical. Permite identificar la pérdida de rendimiento a medida que aumenta el ruido.

Para cuatro clases equilibradas, la selección aleatoria de una modulación corresponde a una precisión esperada de aproximadamente **25 %**. Este valor se utiliza como referencia comparativa y no representa un resultado del modelo.

In [ ]:
filas_clase_snr = []
for clase_id, nombre in enumerate(modulaciones):
    for snr in sorted(np.unique(snr_test)):
        indices = (y_test == clase_id) & (snr_test == snr)
        n = int(indices.sum())
        if n:
            acc = 100 * np.mean(y_pred[indices] == y_test[indices])
            filas_clase_snr.append({"Modulación": nombre, "SNR (dB)": int(snr),
                                    "Señales": n, "Precisión (%)": round(acc, 2)})

por_clase_snr = pd.DataFrame(filas_clase_snr)
por_clase_snr.to_csv("precision_por_clase_y_snr.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
axes[0].plot(resultados_snr["SNR (dB)"], resultados_snr["Precisión (%)"],
             marker="o", linewidth=2, label="Precisión general")
axes[0].axhline(25, linestyle="--", color="gray", label="Azar (25 %)")
axes[0].set_title("Precisión general frente al ruido")
axes[0].set_xlabel("SNR (dB)")
axes[0].set_ylabel("Precisión (%)")
axes[0].set_ylim(0, 105)
axes[0].grid(alpha=0.3)
axes[0].legend()

for nombre in modulaciones:
    parte = por_clase_snr[por_clase_snr["Modulación"] == nombre]
    axes[1].plot(parte["SNR (dB)"], parte["Precisión (%)"], marker="o", label=nombre)
axes[1].set_title("Precisión de cada modulación")
axes[1].set_xlabel("SNR (dB)")
axes[1].set_ylabel("Precisión (%)")
axes[1].set_ylim(0, 105)
axes[1].grid(alpha=0.3)
axes[1].legend()

fig.tight_layout()
fig.savefig("rendimiento_snr_fase3.png", dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
# Criterio orientativo: una precisión de 30 % o menos se considera
# cercana al 25 % esperado por azar. No es una prueba estadística.
cerca_del_azar = resultados_snr[resultados_snr["Precisión (%)"] <= 30]
if len(cerca_del_azar):
    print("En estos niveles el resultado quedó cerca del azar (≤ 30 %):")
    print(cerca_del_azar[["SNR (dB)", "Precisión (%)"]].to_string(index=False))
else:
    print("En los niveles medidos (−5 a 20 dB), la precisión general no cayó al 30 % o menos.")
    print("No se puede afirmar en qué SNR se 'vuelve ciego' sin probar niveles más bajos.")

### 3.4 Análisis de errores

Se calcula el porcentaje de aciertos de cada modulación respecto de su número de muestras, equivalente a su **recall**. La clase con el menor valor se considera la más difícil de reconocer en este conjunto de prueba.

También se identifican las confusiones más frecuentes y se presenta una posible explicación técnica de sus causas, interpretada como una hipótesis y no como una conclusión demostrada únicamente por la matriz.

In [ ]:
resumen_clases = []
for i, nombre in enumerate(modulaciones):
    total = int(matriz[i].sum())
    aciertos = int(matriz[i, i])
    resumen_clases.append({"Modulación": nombre, "Total": total,
                           "Aciertos": aciertos, "Errores": total - aciertos,
                           "Precisión dentro de clase (%)": round(100 * aciertos / total, 2)})

tabla_clases = pd.DataFrame(resumen_clases)
tabla_clases.to_csv("analisis_por_modulacion.csv", index=False)
print("ACIERTO POR MODULACIÓN")
print(tabla_clases.to_string(index=False))

menor = tabla_clases["Precisión dentro de clase (%)"].min()
dificiles = tabla_clases[tabla_clases["Precisión dentro de clase (%)"] == menor]["Modulación"].tolist()
print("\nModulación(es) más difícil(es):", ", ".join(dificiles), f"({menor:.2f}%)")

confusiones = []
for i in range(4):
    for j in range(4):
        if i != j and matriz[i, j] > 0:
            confusiones.append((int(matriz[i, j]), modulaciones[i], modulaciones[j]))
confusiones.sort(reverse=True)

print("\nConfusiones más frecuentes:")
if confusiones:
    for cantidad, real, predicha in confusiones[:3]:
        print(f"  {real} clasificada como {predicha}: {cantidad}")
else:
    print("  No hubo errores en estas señales de prueba.")

hipotesis = {
    "ASK": "Con ruido, los cambios de amplitud pueden quedar ocultos o parecerse a otras formas de onda.",
    "FSK": "El ruido puede dificultar la separación de las dos frecuencias utilizadas para representar bits.",
    "PSK": "Los cambios de fase pueden ser difíciles de distinguir cuando la forma de onda está alterada.",
    "QAM": "Al combinar varios niveles de amplitud y fase, sus símbolos pueden confundirse más fácilmente con ruido."
}
print("\nHipótesis técnica (no es una causa demostrada):")
for nombre in dificiles:
    print(f"{nombre}: {hipotesis[nombre]}")

### Visualización de ejemplos clasificados incorrectamente

In [ ]:
indices_error = np.where(y_pred != y_test)[0]
if len(indices_error) == 0:
    print("No hubo predicciones incorrectas en este conjunto de prueba.")
else:
    mostrar = indices_error[:min(3, len(indices_error))]
    fig, axes = plt.subplots(len(mostrar), 1, figsize=(10, 2.7 * len(mostrar)))
    axes = np.atleast_1d(axes)
    for ax, i in zip(axes, mostrar):
        t = np.arange(128) / 8000 * 1000
        ax.plot(t, X_test[i, :128])
        ax.set_title(f"Real: {modulaciones[y_test[i]]}  |  Predicha: {modulaciones[y_pred[i]]}  |  SNR: {snr_test[i]} dB")
        ax.set_xlabel("Tiempo (ms)")
        ax.set_ylabel("Amplitud")
        ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig("ejemplos_errores_fase3.png", dpi=150, bbox_inches="tight")
    plt.show()

### Resultados finales de la evaluación

Se registra la precisión global sobre las 800 muestras de prueba, la menor precisión observada entre los niveles de SNR, la modulación con más dificultades de reconocimiento y las confusiones ASK/PSK. Las tablas y figuras se exportan para documentar el comportamiento del clasificador.

In [ ]:
peor_snr = resultados_snr.loc[resultados_snr["Precisión (%)"].idxmin()]
resumen_texto = (
    "EL DETECTIVE DE SEÑALES — FASE 3\n"
    "Alumno: Diego Sanchez Gutierrez\n"
    f"Precisión global en prueba: {exactitud*100:.2f}% (800 señales).\n"
    f"ASK confundidas con PSK: {ask_como_psk}; PSK confundidas con ASK: {psk_como_ask}.\n"
    f"Menor precisión por nivel SNR: {peor_snr['Precisión (%)']:.2f}% en {int(peor_snr['SNR (dB)'])} dB.\n"
    f"Modulación(es) con menor precisión: {', '.join(dificiles)} ({menor:.2f}%).\n"
    "La evaluación utiliza únicamente el conjunto de prueba reservado en Fase 2.\n"
)
print(resumen_texto)
with open("conclusiones_fase3.txt", "w", encoding="utf-8") as archivo:
    archivo.write(resumen_texto)

resultados = [
    "matriz_confusion_fase3.png", "matriz_confusion_fase3.csv",
    "precision_por_snr.csv", "precision_por_clase_y_snr.csv",
    "rendimiento_snr_fase3.png", "analisis_por_modulacion.csv",
    "conclusiones_fase3.txt"
]
if os.path.isfile("ejemplos_errores_fase3.png"):
    resultados.append("ejemplos_errores_fase3.png")
with zipfile.ZipFile("Resultados_Fase3.zip", "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    for nombre in resultados:
        zipf.write(nombre)
print("Archivo creado: Resultados_Fase3.zip")